In [ ]:
# ============================================
# LAKEHOUSE PIPELINE — PATH CONFIGURATION
# ============================================
# Set the base_path widget to your Databricks Unity Catalog
# Volume path before running this notebook.
#
# Default: /Volumes/main/lakehouse_pipeline/raw_data
#   Update this default to match your actual catalog name.
#   (Run: SELECT current_catalog() in a SQL notebook to check.)
#
# How to set the value:
#   Interactive  — Edit the widget text box at the top of the notebook.
#   Workflows    — Pass base_path as a job task parameter;
#                  dbutils.widgets picks it up automatically.
#
# See configs/app_config.yaml for full configuration reference.
# ============================================

# Register widget — update the default value to your Volume path
dbutils.widgets.text(
    "base_path",
    "/Volumes/main/lakehouse_pipeline/raw_data",
    "Base Volume Path"
)

BASE_PATH = dbutils.widgets.get("base_path")

print(f"BASE_PATH configured: {BASE_PATH}")

In [0]:
# ============================================
# BRONZE LAYER PIPELINE
# ============================================

from pyspark.sql.functions import *

# ============================================
# CLEAN OLD BRONZE DATA
# ============================================

dbutils.fs.rm(
    f"{BASE_PATH}/bronze/orders",
    recurse=True
)

dbutils.fs.rm(
    f"{BASE_PATH}/bronze/customers",
    recurse=True
)

# ============================================
# READ CUSTOMERS JSON
# ============================================

customers_df = spark.read \
    .option("multiLine", "true") \
    .json(
        f"{BASE_PATH}/customers.json"
    )

# ============================================
# READ ORDERS JSON
# ============================================

orders_df = spark.read \
    .option("multiLine", "true") \
    .json(
        f"{BASE_PATH}/orders.json"
    )

# ============================================
# SHOW DATA
# ============================================

print("CUSTOMERS DATA")
display(customers_df.limit(20))

print("ORDERS DATA")
display(orders_df.limit(20))

# ============================================
# PRINT SCHEMA
# ============================================

customers_df.printSchema()
orders_df.printSchema()

# ============================================
# WRITE BRONZE TABLES
# ============================================

customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        f"{BASE_PATH}/bronze/customers"
    )

orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        f"{BASE_PATH}/bronze/orders"
    )

print("Bronze Layer Created Successfully!")

CUSTOMERS DATA


country,customer_id,name
Australia,1,Mary Phillips
Brazil,2,Janet Bernard
Germany,3,Brandi Gilbert
Germany,4,Timothy Parrish
Singapore,5,Paul Miranda
UK,6,Kristie Bruce MD
USA,7,Kendra Willis
Germany,8,Robert Rubio
Germany,9,Eileen Watson
Singapore,10,Chad Kim


ORDERS DATA


amount,customer_id,order_id,order_time,status
4235.29,6425.0,1,2025-02-26 17:41:00,SUCCESS
-3554.03,5570.0,2,2025-11-29 17:15:00,FAILED
977.88,2643.0,3,2025-11-16 02:05:00,SUCCESS
4298.18,1959.0,4,2025-10-16 17:30:00,SUCCESS
603.99,4695.0,5,2025-09-02 22:49:00,SUCCESS
4365.65,7010.0,6,2025-02-19 14:54:00,SUCCESS
241.2,6543.0,7,2025-06-28 08:29:00,SUCCESS
515.81,9258.0,8,2025-01-28 00:42:00,SUCCESS
2172.64,4874.0,9,2025-03-05 12:42:00,SUCCESS
1975.64,7382.0,10,2025-11-15 21:39:00,SUCCESS


root
 |-- country: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)

root
 |-- amount: double (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_time: string (nullable = true)
 |-- status: string (nullable = true)

Bronze Layer Created Successfully!
